In [17]:
import xarray as xr
import numpy as np
import pandas as pd
import os
import re
from shapely import wkt
from scipy.spatial.distance import cdist
from sklearn.metrics import r2_score
from scipy.stats import pearsonr

from datetime import datetime
from dateutil import parser
import pyproj
import geopandas as gpd
from shapely.geometry import box
from pyproj import CRS, Proj, Transformer

import warnings 
warnings.filterwarnings('ignore')

from tqdm import tqdm

import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F

from skimage.metrics import structural_similarity as compare_ssim
import matplotlib.gridspec as gridspec


import matplotlib.pyplot as plt

In [21]:
# IDW function
def IDW(coords, values, target_coords, p=2):
    # target_coords are the x, y grid cells we want to interpolate onto

    xx, yy = np.meshgrid(target_coords[0], target_coords[1])
    u = np.empty((xx.shape))

    for i in range(xx.shape[0]):
        for j in range(xx.shape[1]):
            current_xy = np.array([[xx[i,j], yy[i,j]]])
            d = cdist(current_xy, np.array(coords), 'euclidean')

            if d.min() == 0:
                u_ij = values[d.argmin()]  # Directly use the index from d.argmin()
            else:
                d = d**p
                w = 1.0 / d
                u_ij = np.sum(w * values) / np.sum(w)

            u[i, j] = u_ij

    return u

In [22]:
# Change 5km to 1km and rerun the following code to get features for 1km training

grid = pd.read_csv('grid_proj_5km.csv')[['x', 'y', 'geometry', 'lon', 'lat']]

grid['geometry'] = grid['geometry'].apply(wkt.loads)

grid_gdf = gpd.GeoDataFrame(grid, geometry='geometry', crs='ESRI:102003')

In [23]:
states = gpd.read_file('us_boundaries/conus_states.zip')
five_states = states.loc[states['STUSPS'].isin(['NY','PA','OH','MD','WV'])]

polygon = box(-84, 37, -74, 43.5)
region = five_states.clip(polygon)
region_proj = region.to_crs('ESRI:102003')

res = 5000 # 

min_x, min_y, max_x, max_y = region_proj.total_bounds

xnew = np.arange(min_x, max_x, res)
ynew = np.arange(min_y, max_y, res)

x_5km, y_5km = np.meshgrid(xnew,ynew)

transformer = Transformer.from_crs("ESRI:102003", "EPSG:4326", always_xy=True)

lon, lat = transformer.transform(x_5km, y_5km)

min_lon = lon.min()
max_lon = lon.max()
min_lat = lat.min()
max_lat = lat.max()

In [24]:
regridded_files = [f for f in os.listdir('PM_AOD/MODIS_AOD_5km_grid/')]
regridded_files.sort()

# Prepare covariates

## Prepare yearly covariates

In [ ]:
for year in range(2004,2024):

    year = str(year)
    
    covariates = pd.read_csv('PM_AOD/covariates/grid_proj_5km_covariates_'+year+'.csv')

    covariates['geometry'] = covariates['geometry'].apply(wkt.loads)

    covariates_gdf = gpd.GeoDataFrame(covariates, geometry='geometry', crs='ESRI:102003')

    columns_to_drop = [col for col in covariates_gdf.columns if re.search('emission', col)]
    covariates_gdf = covariates_gdf.drop(columns=columns_to_drop)
    covariates_gdf.columns

    """NEI Industrial"""

    nei_industrial = pd.read_csv('NEI/Ver 3 - Interpolated data/Industrial/PM2.5_interpolated_emission.csv')
    nei_industrial_year = nei_industrial[['latitude','longitude', year+'_emission']]

    region_emission_proj = gpd.GeoDataFrame(nei_industrial_year, geometry=gpd.points_from_xy(nei_industrial['longitude'],
                                                                            nei_industrial['latitude']), crs='EPSG:4326').to_crs('ESRI:102003')
    region_emission_proj.reset_index(inplace=True,drop=True)
    region_emission_proj.columns = ['latitude','longitude','industry_emission','geometry']

    for buffer_size in [3000, 5000, 10000, 15000, 30000]: 

        print(buffer_size)

        for index in tqdm(covariates_gdf.index):

            buffer = covariates_gdf.loc[index,'geometry'].buffer(buffer_size)

            selected_index = region_emission_proj.sindex.query(buffer)
            selected_gdf = region_emission_proj.loc[selected_index]

            if selected_gdf.shape[0] > 0:

                ll = gpd.clip(selected_gdf, buffer)['industry_emission'].astype(float).sum()
                covariates_gdf.loc[index, 'ind_emission_s'+str(buffer_size)] = ll

    covariates_gdf.to_csv('PM_AOD/covariates/grid_proj_5km_covariates_'+year+'_v1.csv')

# Prepare daily covariates

### AADT

In [26]:
grid_1km = pd.read_csv('grid_proj_1km.csv')
grid_5km = pd.read_csv('grid_proj_5km.csv')

grid_1km['geometry'] = grid_1km['geometry'].apply(wkt.loads)
grid_5km['geometry'] = grid_5km['geometry'].apply(wkt.loads)

gdf_1km = gpd.GeoDataFrame(grid_1km, geometry='geometry', crs="ESRI:102003") 
gdf_5km = gpd.GeoDataFrame(grid_5km, geometry='geometry', crs="ESRI:102003")

def make_grid_polygon(center, size):
    return box(center.x , center.y , center.x + size, center.y + size)

gdf_1km['geometry'] = gdf_1km['geometry'].apply(lambda pt: make_grid_polygon(pt, 1000))
gdf_5km['geometry'] = gdf_5km['geometry'].apply(lambda pt: make_grid_polygon(pt, 5000))

joined = gpd.sjoin(gdf_1km, gdf_5km[['geometry', 'Unnamed: 0']], how="left", predicate='intersects')
joined = joined.rename(columns={'Unnamed: 0_left': 'id_1km', 'Unnamed: 0_right': 'id_5km'})

id_match = joined[['id_1km', 'id_5km']]

In [27]:
month_factor = pd.read_excel('AADT/Emission factor.xlsx', sheet_name=["Month factor"])["Month factor"]
day_factor = pd.read_excel('AADT/Emission factor.xlsx', sheet_name=["Day factor"])["Day factor"]

In [ ]:
regridded_files = [f for f in os.listdir('PM_AOD/MODIS_AOD_5km_grid/')]
regridded_files.sort()

for ds_index in range(len(regridded_files))[5050:]:

    regridded_fname = regridded_files[ds_index]
    epa_fname = regridded_fname[-14:]

    aod = pd.read_csv('PM_AOD/MODIS_AOD_5km_grid/' + regridded_fname)['aod']

    ds = xr.Dataset(
        {
            'aod': (('y', 'x'), aod.values.reshape(x_5km.shape))
        },
        coords={
            'lat': (('y', 'x'), lat),
            'lon': (('y', 'x'), lon),
            'Projected Y': (('y', 'x'), y_5km),
            'Projected X': (('y', 'x'), x_5km),
        }
    )

    ds['aod'] = ds['aod'].fillna(ds['aod'].mean())

    epa_pm = pd.read_csv('PM_AOD/'+epa_fname)

    ds['stations'] = xr.DataArray(np.full_like(ds.aod, np.nan), coords=ds.coords, dims=['y', 'x'])

    for idx, row in epa_pm.iterrows():

        station_lon, station_lat = row['Longitude'], row['Latitude']

        distance = np.sqrt((ds['lat'] - station_lat)**2 + (ds['lon'] - station_lon)**2)

        min_dist_idx = np.unravel_index(np.argmin(distance.values), distance.shape)

        y_index, x_index = min_dist_idx

        if not np.isnan(ds.stations[y_index, x_index]):
            ds.stations[y_index, x_index] = (ds.stations[y_index, x_index] + row['Arithmetic Mean']) / 2
        else:
            ds.stations[y_index, x_index] = row['Arithmetic Mean']

    y_coords, x_coords = np.where(~np.isnan(ds.stations.values))
    known_values = ds.stations.values[y_coords, x_coords]
    coords = list(zip(ds.x[x_coords].values, ds.y[y_coords].values))

    target_coords = (ds.x.values, ds.y.values)
    u = IDW(coords, known_values, target_coords)

    interpolated_data = xr.DataArray(
        u,
        dims=("y", "x"),
        coords={"y": ds.y.values, "x": ds.x.values},
        name="stations_interpolated"
    )

    ds['stations_interpolated'] = interpolated_data

    f = regridded_files[ds_index][-14:]

    year, month, day = f[:4], f[5:7], f[8:10]

    """## Meteo"""

    grid = pd.read_csv('PM_AOD/covariates/grid_proj_5km_covariates_'+year+'_v1.csv')

    grid['geometry'] = grid['geometry'].apply(wkt.loads)

    grid_gdf = gpd.GeoDataFrame(grid, geometry='geometry', crs='ESRI:102003')

    era_month = xr.open_dataset(f'ERA5/{year}{month}.nc')

    daily_mean = era_month.resample(time='1D').mean()

    lats = xr.DataArray(grid_gdf['lat'].astype(float).values, dims='z')
    lons = xr.DataArray(grid_gdf['lon'].astype(float).values, dims='z')

    day_str = f'{year}-{month}-{day}'  
    daily_mean_sel = daily_mean.sel(longitude=lons, latitude=lats, time=day_str, method='nearest')

    for var in ['u10', 'v10', 'd2m', 't2m', 'sp', 'tp']:
        grid_gdf[var] = daily_mean_sel[var].values


    era_month = xr.open_dataset(f'ERA5/{year}{month}_PBLH.nc')

    daily_mean = era_month.resample(time='1D').mean()


    day_str = f'{year}-{month}-{day}' 
    daily_mean_sel = daily_mean.sel(longitude=lons, latitude=lats, time=day_str, method='nearest')

    grid_gdf['blh'] = daily_mean_sel['blh'].values

    """CAMPD"""

    campd_year = pd.read_csv('CAMPD/Daily '+year+'/formatted-daily-'+year+'-PM25.csv')
    campd_year = campd_year.drop_duplicates(subset=['FacilityID', 'UnitID'])
    campd_year = campd_year.groupby(['FacilityID','Latitude','Longitude']).sum().reset_index()

    campd_day = campd_year[['Latitude','Longitude', year+'-'+month+'-'+day]]
    campd_day = campd_day.loc[campd_day[year+'-'+month+'-'+day] != 0]

    region_emission_proj = gpd.GeoDataFrame(campd_day, geometry=gpd.points_from_xy(campd_day['Longitude'],
                                                                            campd_day['Latitude']), crs='EPSG:4326').to_crs('ESRI:102003')
    region_emission_proj.reset_index(inplace=True,drop=True)
    region_emission_proj
    region_emission_proj.columns = ['Latitude','Longitude','campd_emission','geometry']

    buffer_sizes = [3000, 5000, 10000, 15000, 30000]

    region_emission_index = region_emission_proj.sindex

    for buffer_size in buffer_sizes:

        buffers = grid_gdf['geometry'].buffer(buffer_size)

        emissions_sums = []

        for buffer in buffers:
            selected_index = region_emission_index.query(buffer)

            if selected_index.size > 0:
                selected_gdf = region_emission_proj.iloc[selected_index]
                clipped = gpd.clip(selected_gdf, buffer)
                emission_sum = clipped['campd_emission'].astype(float).sum()
            else:
                emission_sum = 0

            emissions_sums.append(emission_sum)

        grid_gdf[f'campd_emission_s{buffer_size}'] = emissions_sums

    """ Well and production """
    try:
    
        well_year = pd.read_csv('Well and Production/PM2.5/'+year+'_emission.csv')
        well_year = well_year.groupby(['Well_ID','Latitude','Longitude']).sum().reset_index()

        well_day = well_year[['Latitude','Longitude', year+'-'+month+'-'+day]]
        well_day = well_day.loc[well_day[year+'-'+month+'-'+day] != 0]

        region_emission_proj = gpd.GeoDataFrame(well_day, geometry=gpd.points_from_xy(well_day['Longitude'],
                                                                                well_day['Latitude']), crs='EPSG:4326').to_crs('ESRI:102003')
        region_emission_proj.reset_index(inplace=True,drop=True)
        region_emission_proj.columns = ['Latitude','Longitude','well_emission','geometry']

        buffer_sizes = [3000, 5000, 10000, 15000, 30000]
        region_emission_index = region_emission_proj.sindex  

        for buffer_size in buffer_sizes:

            buffers = grid_gdf['geometry'].buffer(buffer_size)

            emissions_sums = []

            for buffer in buffers:
                selected_index = region_emission_index.query(buffer)

                if selected_index.size > 0:
                    selected_gdf = region_emission_proj.iloc[selected_index]
                    clipped = gpd.clip(selected_gdf, buffer)
                    emission_sum = clipped['well_emission'].astype(float).sum()
                else:
                    emission_sum = 0

                emissions_sums.append(emission_sum)

            grid_gdf[f'well_emission_s{buffer_size}'] = emissions_sums
            
    except:
        
        for buffer_size in buffer_sizes:
            grid_gdf[f'well_emission_s{buffer_size}'] = 0


    """NEI oil and gas production"""

    try:
        nei_ong_year = pd.read_csv('NEI/Ver 3 - Interpolated data/ONG/PM2.5/ONG_emission_'+year+'.csv')

        month_matcher = {'Jan': '01', 'Feb': '02', 'Mar': '03', 'Apr': '04',
         'May': '05', 'Jun': '06', 'Jul': '07', 'Aug': '08',
         'Sep': '09', 'Oct': '10', 'Nov': '11', 'Dec': '12'}

        nei_ong_year = nei_ong_year.rename(columns={k: v for k, v in month_matcher.items() if k in nei_ong_year.columns})

        nei_ong_year = nei_ong_year.groupby(['facility_id','latitude','longitude']).sum().reset_index()

        nei_ong_day = nei_ong_year[['latitude','longitude', month]]
        nei_ong_day = nei_ong_day.loc[nei_ong_day[month] != 0]

        region_emission_proj = gpd.GeoDataFrame(nei_ong_day, geometry=gpd.points_from_xy(nei_ong_day['longitude'],
                                                                                nei_ong_day['latitude']), crs='EPSG:4326').to_crs('ESRI:102003')
        region_emission_proj.reset_index(inplace=True,drop=True)
        region_emission_proj.columns = ['Latitude','Longitude','ong_emission','geometry']

        buffer_sizes = [3000, 5000, 10000, 15000, 30000]

        region_emission_index = region_emission_proj.sindex

        for buffer_size in buffer_sizes:

            buffers = grid_gdf['geometry'].buffer(buffer_size)

            emissions_sums = []

            for buffer in buffers:
                selected_index = region_emission_index.query(buffer)

                if len(selected_index) > 0:
                    selected_gdf = region_emission_proj.iloc[selected_index]
                    clipped = gpd.clip(selected_gdf, buffer)
                    emission_sum = clipped['ong_emission'].astype(float).sum()
                else:
                    emission_sum = 0

                emissions_sums.append(emission_sum)

            grid_gdf[f'ong_emission_s{buffer_size}'] = emissions_sums

    except:
        for buffer_size in buffer_sizes:
            grid_gdf[f'ong_emission_s{buffer_size}'] = 0

    covariates_full = list(set(grid_gdf.columns.tolist()) - set(['Unnamed: 0','geometry','lat','lon','x','y']))

    for var in covariates_full:
        ds[var] = xr.DataArray(grid_gdf[var].values.reshape(x_5km.shape), coords=ds.coords, dims=['y', 'x'])

    '''AADT'''
    m_factor = month_factor.loc[month_factor['Month'] == int(month), 'Factor'].values[0]

    date_str = f[:10]

    date_obj = datetime.strptime(date_str, '%Y-%m-%d')

    if date_obj.weekday() >= 5:  # 5 = Saturday, 6 = Sunday
        d_factor = day_factor.loc[day_factor['Day'] == 'Weekend', 'Factor'].values[0]
    else:
        d_factor = day_factor.loc[day_factor['Day'] == 'Weekday', 'Factor'].values[0]

    aadt_nox = pd.read_csv('AADT/Emissions_PM2.5_g_per_sec_'+year+'.csv')
    aadt_nox_5km = aadt_nox.merge(id_match, left_on ='grid_ID', right_on='id_1km')
    aadt_nox_5km = aadt_nox_5km.groupby('id_5km').sum().reset_index()[['id_5km',year]]
    grid_gdf_reset = grid_gdf.reset_index()
    merged = grid_gdf_reset.merge(aadt_nox_5km, left_on='index', right_on='id_5km', how='left')
    merged['aadt_pm'] = merged[year]*m_factor*d_factor
    ds['aadt_pm'] = xr.DataArray(merged['aadt_pm'].values.reshape(x_5km.shape), coords=ds.coords, dims=['y', 'x'])

    ds.to_netcdf('PM_AOD/5km_grid_features/'+regridded_fname[-14:-4]+'.nc')